# 2.4 Предварительный анализ очистка и слияние данных

Здесь проверяются структура и пропуски, описывается смысл всех полей, обрабатываются тексты
и собирается таблица наблюдений. Измеренные концентрации не заполняются выдуманными значениями.


In [2]:
import base64
import csv
import hashlib
import html
import io
import json
import re
import sqlite3
import time
import zlib
from contextlib import contextmanager
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
from IPython.display import display, Markdown, HTML, FileLink

LIMIT = 100
TIMINGS = []
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_colwidth', 75)
pd.set_option('display.precision', 3)

@contextmanager
def measured(stage):
    start = time.perf_counter()
    yield
    TIMINGS.append({'Этап': stage, 'Время, мс': (time.perf_counter() - start) * 1000})

def read_sources(raw):
    air = pd.concat([pd.read_csv(io.StringIO(raw[f'{s}.csv']), nrows=48)
                     for s in ['Aotizhongxin', 'Changping']], ignore_index=True)
    air['timestamp'] = pd.to_datetime(air[['year', 'month', 'day', 'hour']]).dt.tz_localize('Asia/Shanghai')
    air['daypart'] = pd.cut(air['hour'], [-1, 5, 11, 17, 23], labels=['Ночь', 'Утро', 'День', 'Вечер'])
    # daypart вычисляется из исходного часа; station и wd взяты из CSV.
    weather_obj = json.loads(raw['weather.json'])
    weather = pd.DataFrame(weather_obj['hourly'])
    weather = weather.rename(columns={'time': 'timestamp'})
    weather['timestamp'] = pd.to_datetime(weather['timestamp']).dt.tz_localize('Asia/Shanghai')
    weather['city'] = 'Beijing'
    weather['weather_group'] = np.select(
        [weather.weather_code.le(1), weather.weather_code.le(3), weather.weather_code.le(48)],
        ['Ясно', 'Облачно', 'Туман'], default='Осадки')
    sectors = np.array(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'])
    weather['wind_sector'] = sectors[((weather.wind_direction_10m.fillna(0) + 22.5) // 45).astype(int) % 8]
    weather.loc[weather.wind_direction_10m.isna(), 'wind_sector'] = pd.NA

    ns = {'wb': 'http://www.worldbank.org'}
    root = ET.fromstring(raw['indicators.xml'])
    if int(root.attrib.get('pages', 1)) != 1:
        raise ValueError('XML содержит несколько страниц; выборку нужно сузить.')
    indicator_rows = []
    for node in root.findall('wb:data', ns):
        value = node.findtext('wb:value', default='', namespaces=ns)
        indicator_rows.append({
            'country_code': node.findtext('wb:countryiso3code', namespaces=ns),
            'country': node.findtext('wb:country', namespaces=ns),
            'year': int(node.findtext('wb:date', namespaces=ns)),
            'indicator': node.find('wb:indicator', ns).attrib['id'],
            'value': float(value) if value else np.nan,
        })
    wb_long = pd.DataFrame(indicator_rows)
    country_rows = []
    for node in ET.fromstring(raw['countries.xml']).findall('wb:country', ns):
        country_rows.append({
            'country_code': node.attrib['id'],
            'region': node.findtext('wb:region', namespaces=ns),
            'income_group_current': node.findtext('wb:incomeLevel', namespaces=ns),
        })
    countries = pd.DataFrame(country_rows)
    wb = wb_long.pivot(index=['country_code', 'country', 'year'], columns='indicator', values='value').reset_index()
    wb.columns.name = None
    wb = wb.rename(columns={'EN.ATM.PM25.MC.M3': 'annual_pm25', 'SP.POP.TOTL': 'population',
                            'SP.URB.TOTL.IN.ZS': 'urban_pct'})
    wb = wb.merge(countries, on='country_code', validate='many_to_one')

    docs = []
    for item in json.loads(raw['abstracts.json'])['resultList']['result'][:8]:
        text = html.unescape(re.sub(r'<[^>]+>', ' ', item.get('abstractText', '')))
        text = re.sub(r'\s+', ' ', text).strip()
        pubtypes = item.get('pubTypeList', {}).get('pubType', [])
        topic = ('Здоровье' if re.search(r'mortality|cardio|biomarker|disease|oxidative|mitochondri', text, re.I)
                 else 'Оценка загрязнения')
        docs.append({
            'doc_id': f"{item['source']}:{item['id']}", 'title': item['title'], 'text': text,
            'journal': item.get('journalInfo', {}).get('journal', {}).get('title', 'Не указан'),
            'language': item.get('language', 'Не указан'), 'topic': topic,
            'publication_type': 'Обзор' if any('review' in s.lower() for s in pubtypes) else 'Исследование',
            'publication_year': int(item['pubYear']), 'char_count': len(text),
            'word_count': len(re.findall(r"[A-Za-z]+(?:[-'][A-Za-z]+)*", text)),
            'author_count': len(item.get('authorList', {}).get('author', [])),
            'country_code': 'CHN', 'city': 'Beijing',
            'url': f"https://europepmc.org/article/{item['source']}/{item['id']}",
        })
    docs = pd.DataFrame(docs)
    counts = {'CSV': len(air), 'JSON API': len(weather),
              'XML': len(wb_long) + len(countries), 'Тексты': len(docs)}
    assert all(0 < n <= LIMIT for n in counts.values()), counts
    return air, weather, wb, docs, wb_long, countries, counts

def clean_merge(air, weather, wb):
    air = air.copy()
    pollutants = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3']
    for column in pollutants:
        air[column] = pd.to_numeric(air[column], errors='coerce')
        air.loc[air[column] < 0, column] = np.nan
    air = air.drop_duplicates(['station', 'timestamp']).sort_values(['timestamp', 'station'])
    air['wind_group'] = pd.cut(air.WSPM, [-np.inf, 2, 4, np.inf], right=False,
                               labels=['Менее 2 м/с', 'От 2 до 4 м/с', 'От 4 м/с'])
    air['country_code'] = 'CHN'
    air['city'] = 'Beijing'
    weather = weather.drop_duplicates(['city', 'timestamp'])
    joined = air.merge(weather.drop(columns='city'), on='timestamp', how='left',
                       validate='many_to_one', indicator='weather_match')
    joined = joined.merge(wb.drop(columns='country'), on=['country_code', 'year'], how='left',
                           validate='many_to_one', indicator='xml_match')
    assert len(joined) == len(air) <= LIMIT
    return joined

def vectorize_documents(docs):
    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer(stop_words='english', max_features=100,
                                ngram_range=(1, 2), min_df=1, sublinear_tf=True)
    matrix = vectorizer.fit_transform(docs.text)
    terms = vectorizer.get_feature_names_out()
    weights = []
    for i, doc_id in enumerate(docs.doc_id):
        values = matrix[i].toarray().ravel()
        for j in values.argsort()[-8:][::-1]:
            if values[j] > 0:
                weights.append({'doc_id': doc_id, 'term': terms[j], 'weight': float(values[j])})
    return vectorizer, matrix, pd.DataFrame(weights)

def profile(frame, name):
    print(f'{name}: {frame.shape[0]} строк, {frame.shape[1]} столбцов')
    return pd.DataFrame({'Тип': frame.dtypes.astype(str), 'Пропуски': frame.isna().sum(),
                         'Доля пропусков, %': frame.isna().mean() * 100,
                         'Различных значений': frame.nunique(dropna=True)})

FIELD_INFO = {
    'No': ('Номер записи внутри CSV; не глобальный ключ', 'номер', 'Идентификатор', 'номинальная', 'Контроль источника'),
    'year': ('Год измерения', 'год', 'Временной ключ', 'интервальная', 'Связь с XML'),
    'month': ('Месяц измерения', 'месяц', 'Время', 'циклическая', 'Границы выборки'),
    'day': ('День месяца', 'день', 'Время', 'интервальная', 'Сравнение дат'),
    'hour': ('Местный час измерения', 'час', 'Время', 'циклическая', 'Суточный профиль'),
    'PM2.5': ('Концентрация частиц до 2,5 мкм', 'мкг/м³', 'Основной отклик', 'отношений', 'Проверка гипотезы'),
    'PM10': ('Концентрация частиц до 10 мкм', 'мкг/м³', 'Дополнительный отклик', 'отношений', 'Сравнение загрязнителей'),
    'NO2': ('Диоксид азота', 'мкг/м³', 'Дополнительный отклик', 'отношений', 'Сравнение загрязнителей'),
    'SO2': ('Диоксид серы', 'мкг/м³', 'Дополнительный отклик', 'отношений', 'Контроль состава воздуха'),
    'CO': ('Оксид углерода', 'мкг/м³', 'Дополнительный отклик', 'отношений', 'Контроль состава воздуха'),
    'O3': ('Озон', 'мкг/м³', 'Дополнительный отклик', 'отношений', 'Контроль состава воздуха'),
    'TEMP': ('Температура у станции из UCI', '°C', 'Фактор', 'интервальная', 'Связь с погодой'),
    'PRES': ('Давление у станции из UCI', 'гПа', 'Фактор', 'отношений', 'Погодный фон'),
    'DEWP': ('Температура точки росы у станции', '°C', 'Фактор', 'интервальная', 'Погодный фон'),
    'RAIN': ('Осадки у станции из UCI', 'мм', 'Фактор', 'отношений', 'Возможное вымывание частиц'),
    'wd': ('Направление ветра у станции из UCI', 'румб', 'Категория', 'номинальная', 'Перенос загрязнения'),
    'WSPM': ('Скорость ветра у станции из UCI', 'м/с', 'Фактор', 'отношений', 'Связь со слабым ветром'),
    'station': ('Название станции', 'название', 'Пространственный ключ', 'номинальная', 'Сравнение территорий'),
    'timestamp': ('Дата и время Asia/Shanghai', 'UTC+08:00', 'Временной ключ', 'интервальная', 'Соединение по часу'),
    'daypart': ('Период суток, вычислен из часа', 'четыре периода', 'Производная категория', 'номинальная', 'Суточный профиль'),
    'city': ('Город наблюдения или тема текста', 'название', 'Пространственный ключ', 'номинальная', 'География'),
    'country_code': ('Код страны ISO3', 'код', 'Ключ', 'номинальная', 'Связь источников'),
    'country': ('Название страны', 'название', 'Категория', 'номинальная', 'Страновой контекст'),
    'temperature_2m': ('Температура ERA5 на высоте 2 м', '°C', 'Внешний фактор', 'интервальная', 'Дополняет станционную погоду'),
    'relative_humidity_2m': ('Относительная влажность ERA5', '%', 'Внешний фактор', 'отношений', 'Дополняет станционную погоду'),
    'pressure_msl': ('Давление ERA5 на уровне моря', 'гПа', 'Внешний фактор', 'отношений', 'Погодный фон'),
    'precipitation': ('Осадки ERA5 за предшествующий час', 'мм', 'Внешний фактор', 'отношений', 'Возможное вымывание частиц'),
    'wind_speed_10m': ('Скорость ветра ERA5 на высоте 10 м', 'м/с', 'Внешний фактор', 'отношений', 'Сопоставление со станцией'),
    'wind_direction_10m': ('Направление ветра ERA5', 'градус', 'Внешний фактор', 'циклическая', 'Перенос загрязнения'),
    'weather_code': ('Код погоды WMO из API', 'код', 'Категория в числовой записи', 'номинальная', 'Группировка погоды'),
    'weather_group': ('Группа погоды, вычислена из WMO', 'категория', 'Производная категория', 'номинальная', 'Фильтр погоды'),
    'wind_sector': ('Восемь секторов направления ветра ERA5', 'румб', 'Производная категория', 'номинальная', 'Фильтр ветра'),
    'wind_group': ('Группа скорости ветра UCI', 'три группы', 'Производная категория', 'порядковая', 'Сравнение слабого и сильного ветра'),
    'annual_pm25': ('Средняя годовая экспозиция населения страны к PM2.5', 'мкг/м³', 'Контекст', 'отношений', 'Не почасовой городской показатель'),
    'population': ('Численность населения страны', 'человек', 'Контекст', 'отношений', 'Масштаб территории'),
    'urban_pct': ('Доля городского населения страны', '%', 'Контекст', 'отношений', 'Фон урбанизации'),
    'region': ('Регион по классификации Всемирного банка', 'название', 'Категория', 'номинальная', 'Страновой контекст'),
    'income_group_current': ('Группа дохода на дату запроса, не историческая', 'название', 'Категория', 'порядковая', 'Описание источника, не фактор 2013 года'),
    'indicator': ('Код показателя Всемирного банка', 'код', 'Ключ', 'номинальная', 'Разворот XML в таблицу'),
    'value': ('Значение; единица зависит от кода indicator', 'согласно indicator', 'Показатель', 'отношений', 'Разворот XML в таблицу'),
    'doc_id': ('Идентификатор публикации Europe PMC', 'код', 'Ключ', 'номинальная', 'Исключение дублей'),
    'title': ('Название публикации', 'текст', 'Описание', 'текстовая', 'Проверка релевантности'),
    'text': ('Аннотация, очищенная от HTML', 'текст', 'Неструктурированный признак', 'текстовая', 'Аннотация и векторизация'),
    'journal': ('Научный журнал', 'название', 'Категория', 'номинальная', 'Происхождение текста'),
    'language': ('Язык аннотации', 'код', 'Категория', 'номинальная', 'Выбор обработки текста'),
    'topic': ('Тема по явному правилу ключевых слов', 'категория', 'Производная аннотация', 'номинальная', 'Группировка текстов'),
    'publication_type': ('Тип публикации, укрупнён из метаданных', 'категория', 'Производная категория', 'номинальная', 'Группировка текстов'),
    'publication_year': ('Год публикации, не год измерений статьи', 'год', 'Время', 'интервальная', 'Библиографический контекст'),
    'char_count': ('Число символов аннотации после очистки', 'символ', 'Производный показатель', 'отношений', 'Объём текста'),
    'word_count': ('Число слов по регулярному выражению', 'слово', 'Производный показатель', 'отношений', 'Объём текста'),
    'author_count': ('Число авторов в ответе API', 'автор', 'Метаданные', 'отношений', 'Описание публикации'),
    'url': ('Ссылка на исходную публикацию', 'URL', 'Происхождение', 'номинальная', 'Проверка источника'),
    'weather_match': ('Результат соединения с погодой', 'категория', 'Контроль', 'номинальная', 'Полнота слияния'),
    'xml_match': ('Результат соединения со страновым контекстом', 'категория', 'Контроль', 'номинальная', 'Полнота слияния'),
}

def dictionary(frame):
    rows = []
    for column in frame.columns:
        meaning, unit, role, scale, purpose = FIELD_INFO[column]
        rows.append({'Поле': column, 'Смысл': meaning, 'Единица': unit,
                     'Роль': role, 'Шкала': scale, 'Для анализа': purpose})
    return pd.DataFrame(rows)

def stage_result(joined):
    mean = joined.groupby('station')['PM2.5'].mean().sort_values()
    daily = joined.groupby(joined.timestamp.dt.date)['PM2.5'].mean()
    corr = joined[['PM2.5', 'WSPM']].corr(method='spearman').iloc[0, 1]
    return {'station_means': mean.to_dict(), 'day_means': {str(k): v for k, v in daily.items()},
            'spearman_wind': float(corr)}


In [3]:
# Снимок исходных ответов с URL, датой получения и ограничением числа записей.
# Сжатие уменьшает размер файла; искусственные измерения не добавляются.
RAW = json.loads(zlib.decompress(base64.b64decode('...')).decode('utf-8'))
with measured('Чтение CSV, JSON, XML и текстов'):
    air, weather, wb, docs, wb_long, countries, counts = read_sources(RAW)
print('Записей по источникам:', counts)


Записей по источникам: {'CSV': 96, 'JSON API': 48, 'XML': 30, 'Тексты': 8}


## Структура и качество источников

В профиле приведены физические типы, число пропусков и число различных значений.
Константные поля остаются в описании, но не используются как объяснение различий между станциями.


In [4]:
with measured('Профилирование четырёх источников'):
    for name, frame in [('CSV', air), ('JSON погоды', weather), ('XML после разворота', wb), ('Тексты', docs)]:
        display(Markdown(f'### {name}'))
        display(profile(frame, name))
        print('Полных дубликатов:', frame.astype(str).duplicated().sum())


### CSV

CSV: 96 строк, 20 столбцов


,Тип,Пропуски,"Доля пропусков, %",Различных значений
No,int64,0,0.000,48
year,int64,0,0.000,1
month,int64,0,0.000,1
day,int64,0,0.000,2
hour,int64,0,0.000,24
PM2.5,int64,0,0.000,39
PM10,int64,0,0.000,51
SO2,float64,1,1.042,44
NO2,float64,1,1.042,58
CO,float64,1,1.042,16


Полных дубликатов: 0


### JSON погоды

JSON погоды: 48 строк, 11 столбцов


... (6 outputs left out)

### Тексты

Тексты: 8 строк, 14 столбцов


,Тип,Пропуски,"Доля пропусков, %",Различных значений
doc_id,str,0,0.0,8
title,str,0,0.0,8
text,str,0,0.0,8
journal,str,0,0.0,8
language,str,0,0.0,1
topic,str,0,0.0,2
publication_type,str,0,0.0,2
publication_year,int64,0,0.0,1
char_count,int64,0,0.0,8
word_count,int64,0,0.0,8


Полных дубликатов: 0


In [5]:
display(air[['PM2.5', 'PM10', 'NO2', 'TEMP', 'WSPM']].describe().T)
display(air[['station', 'timestamp', 'PM2.5', 'PM10', 'NO2', 'wd']].head(5))
display(weather.head(3))
display(wb.head(3))
display(docs[['doc_id', 'journal', 'topic', 'word_count', 'author_count']].head(3))


,count,mean,std,min,25%,50%,75%,max
PM2.5,96.0,19.073,21.062,3.0,3.000,10.0,26.75,93.0
PM10,96.0,27.906,27.634,3.0,7.750,18.0,34.25,120.0
NO2,95.0,41.137,28.529,6.0,14.000,39.0,61.50,110.0
TEMP,96.0,0.606,2.712,-4.5,-1.625,0.6,2.35,6.2
WSPM,96.0,1.949,1.274,0.0,1.000,1.7,2.80,5.6


,station,timestamp,PM2.5,PM10,NO2,wd
0,Aotizhongxin,2013-03-01 00:00:00+08:00,4,4,7.0,NNW
1,Aotizhongxin,2013-03-01 01:00:00+08:00,8,8,7.0,N
2,Aotizhongxin,2013-03-01 02:00:00+08:00,7,7,10.0,NNW
3,Aotizhongxin,2013-03-01 03:00:00+08:00,6,6,11.0,NW
4,Aotizhongxin,2013-03-01 04:00:00+08:00,3,3,12.0,N


,timestamp,temperature_2m,relative_humidity_2m,pressure_msl,precipitation,wind_speed_10m,wind_direction_10m,weather_code,city,weather_group,wind_sector
0,2013-03-01 00:00:00+08:00,-0.6,25,1028.3,0.0,4.65,332,3,Beijing,Облачно,NW
1,2013-03-01 01:00:00+08:00,-1.2,24,1028.8,0.0,4.98,338,3,Beijing,Облачно,N
2,2013-03-01 02:00:00+08:00,-1.2,23,1029.6,0.0,4.31,338,1,Beijing,Ясно,N


,country_code,country,year,annual_pm25,population,urban_pct,region,income_group_current
0,CHN,China,2013,51.750,1.363e+09,54.49,East Asia & Pacific,Upper middle income
1,CHN,China,2014,52.510,1.372e+09,55.75,East Asia & Pacific,Upper middle income
2,CHN,China,2015,49.473,1.380e+09,57.33,East Asia & Pacific,Upper middle income


,doc_id,journal,topic,word_count,author_count
0,MED:23320082,PloS one,Оценка загрязнения,200,5
1,MED:23321859,Journal of exposure science & environmental epidemiology,Здоровье,267,14
2,MED:23358615,Annals of hematology,Здоровье,296,12


## Словарь атрибутов

Числовой код погоды считается категорией. Температура имеет интервальную шкалу,
концентрации и скорость ветра имеют шкалу отношений. Час и угол направления ветра цикличны.
Координаты погодной ячейки не используются как точные координаты постов мониторинга.


In [6]:
for name, frame in [('Измерения CSV', air), ('Погода JSON', weather), ('Контекст XML', wb), ('Аннотации', docs)]:
    display(Markdown(f'### {name}'))
    display(dictionary(frame))
# В исходном XML value зависит от indicator; единица задаётся кодом показателя.
display(dictionary(wb_long))


### Измерения CSV

,Поле,Смысл,Единица,Роль,Шкала,Для анализа
0,No,Номер записи внутри CSV; не глобальный ключ,номер,Идентификатор,номинальная,Контроль источника
1,year,Год измерения,год,Временной ключ,интервальная,Связь с XML
2,month,Месяц измерения,месяц,Время,циклическая,Границы выборки
3,day,День месяца,день,Время,интервальная,Сравнение дат
4,hour,Местный час измерения,час,Время,циклическая,Суточный профиль
5,PM2.5,"Концентрация частиц до 2,5 мкм",мкг/м³,Основной отклик,отношений,Проверка гипотезы
6,PM10,Концентрация частиц до 10 мкм,мкг/м³,Дополнительный отклик,отношений,Сравнение загрязнителей
7,SO2,Диоксид серы,мкг/м³,Дополнительный отклик,отношений,Контроль состава воздуха
8,NO2,Диоксид азота,мкг/м³,Дополнительный отклик,отношений,Сравнение загрязнителей
9,CO,Оксид углерода,мкг/м³,Дополнительный отклик,отношений,Контроль состава воздуха


### Погода JSON

,Поле,Смысл,Единица,Роль,Шкала,Для анализа
0,timestamp,Дата и время Asia/Shanghai,UTC+08:00,Временной ключ,интервальная,Соединение по часу
1,temperature_2m,Температура ERA5 на высоте 2 м,°C,Внешний фактор,интервальная,Дополняет станционную погоду
2,relative_humidity_2m,Относительная влажность ERA5,%,Внешний фактор,отношений,Дополняет станционную погоду
3,pressure_msl,Давление ERA5 на уровне моря,гПа,Внешний фактор,отношений,Погодный фон
4,precipitation,Осадки ERA5 за предшествующий час,мм,Внешний фактор,отношений,Возможное вымывание частиц
5,wind_speed_10m,Скорость ветра ERA5 на высоте 10 м,м/с,Внешний фактор,отношений,Сопоставление со станцией
6,wind_direction_10m,Направление ветра ERA5,градус,Внешний фактор,циклическая,Перенос загрязнения
7,weather_code,Код погоды WMO из API,код,Категория в числовой записи,номинальная,Группировка погоды
8,city,Город наблюдения или тема текста,название,Пространственный ключ,номинальная,География
9,weather_group,"Группа погоды, вычислена из WMO",категория,Производная категория,номинальная,Фильтр погоды


### Контекст XML

,Поле,Смысл,Единица,Роль,Шкала,Для анализа
0,country_code,Код страны ISO3,код,Ключ,номинальная,Связь источников
1,country,Название страны,название,Категория,номинальная,Страновой контекст
2,year,Год измерения,год,Временной ключ,интервальная,Связь с XML
3,annual_pm25,Средняя годовая экспозиция населения страны к PM2.5,мкг/м³,Контекст,отношений,Не почасовой городской показатель
4,population,Численность населения страны,человек,Контекст,отношений,Масштаб территории
5,urban_pct,Доля городского населения страны,%,Контекст,отношений,Фон урбанизации
6,region,Регион по классификации Всемирного банка,название,Категория,номинальная,Страновой контекст
7,income_group_current,"Группа дохода на дату запроса, не историческая",название,Категория,порядковая,"Описание источника, не фактор 2013 года"


### Аннотации

,Поле,Смысл,Единица,Роль,Шкала,Для анализа
0,doc_id,Идентификатор публикации Europe PMC,код,Ключ,номинальная,Исключение дублей
1,title,Название публикации,текст,Описание,текстовая,Проверка релевантности
2,text,"Аннотация, очищенная от HTML",текст,Неструктурированный признак,текстовая,Аннотация и векторизация
3,journal,Научный журнал,название,Категория,номинальная,Происхождение текста
4,language,Язык аннотации,код,Категория,номинальная,Выбор обработки текста
5,topic,Тема по явному правилу ключевых слов,категория,Производная аннотация,номинальная,Группировка текстов
6,publication_type,"Тип публикации, укрупнён из метаданных",категория,Производная категория,номинальная,Группировка текстов
7,publication_year,"Год публикации, не год измерений статьи",год,Время,интервальная,Библиографический контекст
8,char_count,Число символов аннотации после очистки,символ,Производный показатель,отношений,Объём текста
9,word_count,Число слов по регулярному выражению,слово,Производный показатель,отношений,Объём текста


,Поле,Смысл,Единица,Роль,Шкала,Для анализа
0,country_code,Код страны ISO3,код,Ключ,номинальная,Связь источников
1,country,Название страны,название,Категория,номинальная,Страновой контекст
2,year,Год измерения,год,Временной ключ,интервальная,Связь с XML
3,indicator,Код показателя Всемирного банка,код,Ключ,номинальная,Разворот XML в таблицу
4,value,Значение; единица зависит от кода indicator,согласно indicator,Показатель,отношений,Разворот XML в таблицу


## Правила очистки

Временные компоненты CSV объединяются в дату с часовым поясом Asia/Shanghai.
Отрицательные концентрации заменяются пропусками; одинаковые ключи станция-час удаляются.
Большие положительные значения сохраняются: скачок загрязнения может быть реальным событием.
Пропуски в целевых концентрациях не интерполируются и не заполняются средним.


In [7]:
with measured('Очистка и слияние'):
    merged = clean_merge(air, weather, wb)
pollutants = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3']
quality = pd.DataFrame({
    'Отрицательных до очистки': (air[pollutants] < 0).sum(),
    'Пропусков до очистки': air[pollutants].isna().sum(),
    'Пропусков после очистки': merged[pollutants].isna().sum(),
})
display(quality)
print('Повторных ключей до очистки:', air.duplicated(['station', 'timestamp']).sum())
print('Пропусков во всех полях CSV:', int(air.isna().sum().sum()))
print('Строк после очистки и слияния:', len(merged))
print('Полных совпадений с погодой:', int(merged.weather_match.eq('both').sum()))
print('Полных совпадений с XML:', int(merged.xml_match.eq('both').sum()))
assert len(merged) == 96
assert not merged.duplicated(['station', 'timestamp']).any()
assert merged.weather_match.eq('both').all()
assert merged.xml_match.eq('both').all()


,Отрицательных до очистки,Пропусков до очистки,Пропусков после очистки
PM2.5,0,0,0
PM10,0,0,0
SO2,0,1,1
NO2,0,1,1
CO,0,1,1
O3,0,1,1


Повторных ключей до очистки: 0
Пропусков во всех полях CSV: 4
Строк после очистки и слияния: 96
Полных совпадений с погодой: 96
Полных совпадений с XML: 96


## Как соединены разные форматы

CSV и JSON соединены по местному часу. JSON содержит одну городскую точку,
поэтому одна погодная запись относится к обеим станциям; это соединение «многие к одному».
XML соединён по коду страны и году. Страновые значения повторяются как справочный фон
и исключаются из почасовой проверки гипотезы.

Тексты связываются с исследованием по теме и городу через отдельный справочник.
Год публикации не приравнивается к году измерений, описанных в статье.
Из восьми статей не создаются восемь копий каждого измерения.


In [8]:
display(merged[['station', 'timestamp', 'PM2.5', 'PM10', 'NO2', 'WSPM',
                'temperature_2m', 'relative_humidity_2m', 'annual_pm25']].head(6))
print('Полей объединённой таблицы:', len(merged.columns))
display(dictionary(merged[['wind_group', 'weather_match', 'xml_match']]))


,station,timestamp,PM2.5,PM10,NO2,WSPM,temperature_2m,relative_humidity_2m,annual_pm25
0,Aotizhongxin,2013-03-01 00:00:00+08:00,4.0,4.0,7.0,4.4,-0.6,25,51.75
1,Changping,2013-03-01 00:00:00+08:00,3.0,6.0,7.0,0.5,-0.6,25,51.75
2,Aotizhongxin,2013-03-01 01:00:00+08:00,8.0,8.0,7.0,4.7,-1.2,24,51.75
3,Changping,2013-03-01 01:00:00+08:00,3.0,3.0,6.0,0.7,-1.2,24,51.75
4,Aotizhongxin,2013-03-01 02:00:00+08:00,7.0,7.0,10.0,5.6,-1.2,23,51.75
5,Changping,2013-03-01 02:00:00+08:00,3.0,3.0,13.0,0.2,-1.2,23,51.75


Полей объединённой таблицы: 39


,Поле,Смысл,Единица,Роль,Шкала,Для анализа
0,wind_group,Группа скорости ветра UCI,три группы,Производная категория,порядковая,Сравнение слабого и сильного ветра
1,weather_match,Результат соединения с погодой,категория,Контроль,номинальная,Полнота слияния
2,xml_match,Результат соединения со страновым контекстом,категория,Контроль,номинальная,Полнота слияния


## Аннотация и векторизация текстов

HTML удаляется, пробелы нормализуются. Правило с терминами mortality, cardio, biomarker,
disease, oxidative и mitochondri относит аннотацию к теме «Здоровье»; остальные получают
тему «Оценка загрязнения». Это автоматическая разметка по словарю, а не экспертная оценка.

TF-IDF переводит текст в числовые веса слов и пар слов. Частые английские служебные слова удаляются,
словарь ограничен 100 признаками. Для каждой аннотации сохраняются восемь наиболее весомых терминов.
Эти признаки характеризуют содержание публикаций и не подменяют измерения загрязнения.


In [9]:
with measured('Аннотация и TF-IDF текстов'):
    vectorizer, text_matrix, term_weights = vectorize_documents(docs)
display(docs[['doc_id', 'title', 'topic', 'publication_type', 'char_count', 'word_count', 'author_count']])
print('Размер матрицы TF-IDF:', text_matrix.shape)
density = text_matrix.nnz / (text_matrix.shape[0] * text_matrix.shape[1])
print(f'Доля ненулевых элементов: {density:.1%}')
top_terms = term_weights.groupby('doc_id', sort=False).term.agg(', '.join).rename('Термины')
display(docs[['doc_id', 'title']].merge(top_terms, on='doc_id'))
assert text_matrix.shape[0] == len(docs) == 8
assert text_matrix.shape[1] <= 100


,doc_id,title,topic,publication_type,char_count,word_count,author_count
0,MED:23320082,Estimation of citywide air pollution in Beijing.,Оценка загрязнения,Исследование,1434,200,5
1,MED:23321859,Malondialdehyde in exhaled breath condensate and urine as a biomarker o...,Здоровье,Исследование,1907,267,14
2,MED:23358615,Changes of plasma vWF level in response to the improvement of air quali...,Здоровье,Исследование,1977,296,12
3,MED:23375554,Size-fractioned particulate air pollution and cardiovascular emergency ...,Здоровье,Исследование,3531,463,14
4,MED:23646463,Cardiorespiratory biomarker responses in healthy young adults to drasti...,Здоровье,Исследование,6588,864,16
5,MED:23434817,Systematic review of Chinese studies of short-term exposure to air poll...,Здоровье,Обзор,1345,180,10
6,MED:23624010,The association between diurnal temperature range and emergency room ad...,Здоровье,Исследование,1719,249,9
7,MED:23628000,Inhalable particulate matter and mitochondrial DNA copy number in highl...,Здоровье,Исследование,1811,237,12


Размер матрицы TF-IDF: (8, 100)
Доля ненулевых элементов: 38.5%


,doc_id,title,Термины
0,MED:23320082,Estimation of citywide air pollution in Beijing.,"pm, mass, daily, 10, concentrations, quality, air quality, data"
1,MED:23321859,Malondialdehyde in exhaled breath condensate and urine as a biomarker o...,"mda, ebc, increases, olympic, pollutants, oxidative, stress, oxidative ..."
2,MED:23358615,Changes of plasma vWF level in response to the improvement of air quali...,"plasma vwf, vwf, stages, vwf level, plasma, olympic games, games, olympic"
3,MED:23375554,Size-fractioned particulate air pollution and cardiovascular emergency ...,"particles, cardiovascular emergency, room visits, visits, size, number ..."
4,MED:23646463,Cardiorespiratory biomarker responses in healthy young adults to drasti...,"olympics, olympics period, ebc, nitrite, nitrate, ebc nitrite, biomarke..."
5,MED:23434817,Systematic review of Chinese studies of short-term exposure to air poll...,"mortality, pm2, increase, exposure, 10, air, air quality, quality"
6,MED:23624010,The association between diurnal temperature range and emergency room ad...,"dtr, admissions, er, er admissions, elderly, lag, increase, effect"
7,MED:23628000,Inhalable particulate matter and mitochondrial DNA copy number in highl...,"mtdnacn, ambient, particle, blood, 95 ci, pm, measured, day average"


In [10]:
text_links = docs[['doc_id', 'city', 'country_code', 'publication_year', 'topic']].copy()
print('Таблица тематической связи документов с городом:')
display(text_links)
print('Ключи измерений сохранены; тексты не размножают факты:', len(merged))
display(pd.DataFrame(TIMINGS))


Таблица тематической связи документов с городом:


,doc_id,city,country_code,publication_year,topic
0,MED:23320082,Beijing,CHN,2013,Оценка загрязнения
1,MED:23321859,Beijing,CHN,2013,Здоровье
2,MED:23358615,Beijing,CHN,2013,Здоровье
3,MED:23375554,Beijing,CHN,2013,Здоровье
4,MED:23646463,Beijing,CHN,2013,Здоровье
5,MED:23434817,Beijing,CHN,2013,Здоровье
6,MED:23624010,Beijing,CHN,2013,Здоровье
7,MED:23628000,Beijing,CHN,2013,Здоровье


Ключи измерений сохранены; тексты не размножают факты: 96


,Этап,"Время, мс"
0,"Чтение CSV, JSON, XML и текстов",17.128
1,Профилирование четырёх источников,31.282
2,Очистка и слияние,10.906
3,Аннотация и TF-IDF текстов,1158.016


## Результат пункта 2.4

Подготовлены 96 почасовых измерений с погодой и страновым контекстом, восемь размеченных аннотаций
и их матрица TF-IDF. Контроль ключей и кратности соединений выполнен в коде.
Отсутствие ошибок в этом фрагменте не означает отсутствие ошибок в полном наборе данных.
